In [ ]:
# --- MASTER INTERACTIVE GUI & DATA VISUALIZATION DASHBOARD ---
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Ensure matplotlib displays properly inside the notebook layout
%matplotlib inline

# Initialize the global configuration variables
DB_NAME = "payroll_ledger.db"
CSV_BACKUP_NAME = "payroll_history_backup.csv"
EMPLOYEE_ID = "EE-001"

# Load the exact regulatory rates compiled previously
SS_WAGE_BASE = 184500.00      
FUTA_WAGE_BASE = 7000.00      
NM_SUI_WAGE_BASE = 34800.00   
SS_RATE, MEDICARE_RATE, FUTA_NET_RATE, NM_SUI_RATE = 0.062, 0.0145, 0.006, 0.034
NM_WC_FEE_EE_ONLY = 2.55
#NM_WC_FEE_PER_PAY = (2.55 * 4) / periods_slider.value # Quarterly $2.55 fee normalized per pay period EE only (ER must also match)

FED_BRACKETS = [(12400, 0.10), (50400, 0.12), (105700, 0.22), (201775, 0.24), (256225, 0.32), (640600, 0.35), (float('inf'), 0.37)]
NM_BRACKETS = [(8050, 0.00), (13550, 0.015), (20550, 0.032), (24550, 0.032), (33000, 0.043), (41000, 0.043), (58000, 0.047), (74000, 0.047), (102100, 0.047), (116100, 0.049), (217500, 0.049), (331100, 0.049), (float('inf'), 0.059)]

# 1. Setup the GUI Input Form Controls
salary_slider = widgets.IntSlider(value=60000, min=30000, max=250000, step=1000, description='Annual Base:')
periods_slider = widgets.IntSlider(value=12, min=1, max=52, step=1, description='Pay Periods:')
hip_slider = widgets.FloatSlider(value=200.0, min=0.0, max=3000.0, step=10.0, description='HIP/DIP Pmt/Reimb:')
retire_slider = widgets.FloatSlider(value=0.0, min=0.0, max=25.0, step=0.5, description='401k Def %:')
date_picker = widgets.DatePicker(value=pd.to_datetime('2026-06-15').date(), description='Pay Date:')
save_button = widgets.Button(description="Commit Pay Run", button_style='success', icon='check')
output_panel = widgets.Output()

# Ensure NM_WC_FEE_PER_PAY updates dynamically when the number of pay periods changes
# TODO Actually NM for a single EE S-Corp Owner does not require WC Fee
periods_slider.observe(lambda change: globals().update({'NM_WC_FEE_PER_PAY': (2.55 * 4) / periods_slider.value}), 'value')  

# 2. Mathematical Engines
def calc_capped(gross, ytd, cap, rate):
    if ytd >= cap: return 0.0
    return min(gross, cap - ytd) * rate

def calc_progressive(gross, periods, brackets):
    annual = gross * periods
    tax = 0.0
    prev = 0.0
    for limit, rate in brackets:
        if annual > limit:
            tax += (limit - prev) * rate
            prev = limit
        else:
            tax += (annual - prev) * rate
            break
    return tax / periods

# 3. Dynamic Dashboard Rendering Engine
def update_dashboard(*args):
    with output_panel:
        clear_output(wait=True)
        
        # Calculate current automated YTD values from live DB
        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()
        cursor.execute("SELECT SUM(gross_wages) FROM payroll_ledger WHERE employee_id = ?", (EMPLOYEE_ID,))
        res = cursor.fetchone()
        ytd_prior = float(res[0]) if res[0] is not None else 0.00
        conn.close()
        
        # Pull parameters from current GUI sliders
        base_gross = salary_slider.value / periods_slider.value
        hip_pmt_reimb = hip_slider.value
        current_gross = base_gross + hip_pmt_reimb
        deduction_401k = base_gross * (retire_slider.value / 100.0)

        # S-CORP STRUCTURAL SEPARATION: 
        # FICA applies to base gross pay only. HIP is exempt.
        fica_subject_gross = base_gross 
        # Income Tax applies to base gross AND health premiums minus pre-tax deductions
        income_tax_subject_gross = base_gross + hip_pmt_reimb - deduction_401k
        
        # Debug prints to verify the structural separation
        print(f"FICA Subject Gross: ${fica_subject_gross:,.2f}")
        print(f"Income Tax Subject Gross: ${income_tax_subject_gross:,.2f}")    
        # Process deductions and progressive taxes
        ee_ss = calc_capped(fica_subject_gross, ytd_prior, SS_WAGE_BASE, SS_RATE)
        ee_med = fica_subject_gross * MEDICARE_RATE
        ee_fed = calc_progressive(income_tax_subject_gross, periods_slider.value, FED_BRACKETS)
        ee_nm = calc_progressive(income_tax_subject_gross, periods_slider.value, NM_BRACKETS)

        nm_wc_fee_per_pay = (2.55 * 4) / periods_slider.value
        
        # Calculate Owner Take-Home Net Cash
        total_ee_taxes = ee_ss + ee_med + ee_fed + ee_nm + nm_wc_fee_per_pay
        net_pay = current_gross - deduction_401k - total_ee_taxes
        
        #net_pay = current_gross - deduction_401k - (ee_ss + ee_med + ee_fed + ee_nm + NM_WC_FEE_PER_PAY)
        
        # --- REMAPPED STRUCTURAL DATA DESIGN ---
        categories = ['Net Take-Home', 'Pre-Tax 401k', 'Fed Income Tax', 'NM State Tax', 'FICA taxes']
        sizes = [net_pay, deduction_401k, ee_fed, ee_nm, (ee_ss + ee_med)]
        colors = ['#2ecc71', '#3498db', '#e74c3c', '#f1c40f', '#95a5a6']
        
        # 1. Dynamically append the dollar values under the text category names using newlines (\n)
        combined_labels = []
        for cat, val in zip(categories, sizes):
            combined_labels.append(f"{cat}\n(${val:,.2f})")
        
        # 2. Filter out $0 lines so zeroed brackets don't render clutter text
        labels = [l for i, l in enumerate(combined_labels) if sizes[i] > 0]
        colors = [c for i, c in enumerate(colors) if sizes[i] > 0]
        sizes = [s for s in sizes if s > 0]
        
        # 3. Generate Visual Layout
        fig, ax = plt.subplots(figsize=(8.5, 4.5)) # Slightly enlarged to allow full text padding
        
        ax.pie(
            sizes, 
            labels=labels, 
            colors=colors,
            autopct='%1.1f%%',       # Keep clean percentages isolated inside the wedge paths
            startangle=140, 
            pctdistance=0.65,        # Positions percent tags safely within the slices
            labeldistance=1.15,      # Pushes text labels and dollar amounts outside the circle line
            textprops={'fontsize': 9, 'weight': 'bold'}
        )
        
        ax.set_title(f"Paycheck Allocation Breakdown (Gross Period Total: ${current_gross:,.2f})", fontsize=11, weight='bold', pad=20)
        plt.tight_layout()
        plt.show()
        
        # Print a scannable summary under the chart
        print(f"💰 Employee Net Take-Home: ${net_pay:,.2f}  |  🏦 401(k) Deferral: ${deduction_401k:,.2f}")
        print(f"🛑 Combined Taxes Withheld: ${(current_gross - deduction_401k - net_pay):,.2f} | Prior YTD Tracked: ${ytd_prior:,.2f}")

# 4. Action Handle: Saving to SQLite and backing up to CSV via the button
def on_save_clicked(b):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("SELECT SUM(gross_wages) FROM payroll_ledger WHERE employee_id = ?", (EMPLOYEE_ID,))
    res = cursor.fetchone()
    ytd_prior = float(res[0]) if res[0] is not None else 0.00
    
    base_gross = salary_slider.value / periods_slider.value
    stipend = hip_slider.value
    current_gross = base_gross + stipend
    deduction_401k = base_gross * (retire_slider.value / 100.0)
    
    ee_ss = calc_capped(current_gross, ytd_prior, SS_WAGE_BASE, SS_RATE)
    ee_med = current_gross * MEDICARE_RATE
    ee_fed = calc_progressive(current_gross - deduction_401k, periods_slider.value, FED_BRACKETS)
    ee_nm = calc_progressive(current_gross - deduction_401k, periods_slider.value, NM_BRACKETS)
    net_pay = current_gross - deduction_401k - (ee_ss + ee_med + ee_fed + ee_nm + NM_WC_FEE_PER_PAY)
    
    cursor.execute("""
    INSERT INTO payroll_ledger (
        employee_id, pay_period_ending, gross_wages, pre_tax_401k, ee_ss, ee_medicare, ee_fed_tax, ee_nm_tax, ee_wc_fee, net_pay,
        er_ss, er_medicare, er_futa, er_nm_sui, er_wc_fee, total_company_cost
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        EMPLOYEE_ID, str(date_picker.value), current_gross, deduction_401k, ee_ss, ee_med, ee_fed, ee_nm, NM_WC_FEE_PER_PAY, net_pay,
        ee_ss, ee_med, calc_capped(current_gross, ytd_prior, FUTA_WAGE_BASE, FUTA_NET_RATE), calc_capped(current_gross, ytd_prior, NM_SUI_WAGE_BASE, NM_SUI_RATE), NM_WC_FEE_PER_PAY, (current_gross + ee_ss + ee_med + NM_WC_FEE_PER_PAY)
    ))
    conn.commit()
    df = pd.read_sql_query("SELECT * FROM payroll_ledger", conn)
    df.to_csv(CSV_BACKUP_NAME, index=False)
    conn.close()
    
    update_dashboard()
    with output_panel:
        print(f"\n✅ SUCCESS: Payroll for period ending {date_picker.value} committed to SQLite Database and backed up to CSV!")

# Link listeners to trigger instant graph refreshes when adjusting elements
salary_slider.observe(update_dashboard, 'value')
periods_slider.observe(update_dashboard, 'value')
hip_slider.observe(update_dashboard, 'value')
retire_slider.observe(update_dashboard, 'value')
date_picker.observe(update_dashboard, 'value')
save_button.on_click(on_save_clicked)

# Render the layout components to the user interface
input_form = widgets.VBox([salary_slider, periods_slider, hip_slider, retire_slider, date_picker, save_button])
dashboard_layout = widgets.HBox([input_form, output_panel])
display(dashboard_layout)

# Fire initial update to render the default view upon load
update_dashboard()
